# Taller resuelto: Selección de características

## Objetivo
Aplicar técnicas de selección de variables para mejorar interpretabilidad, reducir complejidad y comparar desempeño.

## Bloque 1. Importar librerías

Cargaremos herramientas para datos, selección estadística, modelos y evaluación.

In [ ]:
# Importamos NumPy.
import numpy as np

# Importamos Pandas.
import pandas as pd

# Importamos Matplotlib.
import matplotlib.pyplot as plt

# Importamos dataset.
from sklearn.datasets import load_breast_cancer

# Importamos separación de datos.
from sklearn.model_selection import train_test_split

# Importamos escalador.
from sklearn.preprocessing import StandardScaler

# Importamos SelectKBest.
from sklearn.feature_selection import SelectKBest, f_classif

# Importamos Random Forest.
from sklearn.ensemble import RandomForestClassifier

# Importamos regresión logística.
from sklearn.linear_model import LogisticRegression

# Importamos accuracy.
from sklearn.metrics import accuracy_score


### Interpretación

La selección de características permite reducir variables, mejorar interpretación y controlar ruido.

## Bloque 2. Cargar dataset

Usaremos breast cancer como problema de clasificación con múltiples variables numéricas.

In [ ]:
# Cargamos datos.
data = load_breast_cancer()

# Creamos DataFrame.
df = pd.DataFrame(data.data, columns=data.feature_names)

# Agregamos target.
df["target"] = data.target

# Mostramos primeras filas.
df.head()


### Interpretación

Cada fila representa una observación clínica y cada columna una característica numérica. El objetivo es seleccionar las más relevantes.

## Bloque 3. Separar X e y

Definiremos variables predictoras y variable objetivo.

In [ ]:
# Definimos X sin la columna target.
X = df.drop("target", axis=1)

# Definimos y como columna objetivo.
y = df["target"]

# Mostramos dimensiones.
print("Shape de X:", X.shape)
print("Shape de y:", y.shape)


### Interpretación

`X` contiene las variables candidatas y `y` la respuesta. La selección buscará reducir X conservando información útil.

## Bloque 4. Correlación con el objetivo

Exploraremos qué variables se relacionan más con `target`.

In [ ]:
# Calculamos correlación de cada variable con target.
correlations = df.corr(numeric_only=True)["target"].drop("target").sort_values(key=abs, ascending=False)

# Mostramos top 10.
print(correlations.head(10))

# Creamos figura.
plt.figure(figsize=(10,5))

# Graficamos top 10.
correlations.head(10).plot(kind="bar")

# Agregamos título.
plt.title("Top 10 variables por correlación con target")

# Rotamos etiquetas.
plt.xticks(rotation=45, ha="right")

# Mostramos gráfica.
plt.show()


### Interpretación

La correlación ayuda a identificar variables relacionadas linealmente con el objetivo, pero no prueba causalidad ni captura relaciones complejas.

## Bloque 5. SelectKBest

Seleccionaremos las 10 mejores variables usando ANOVA F-value.

In [ ]:
# Creamos selector para elegir 10 variables.
selector = SelectKBest(score_func=f_classif, k=10)

# Ajustamos selector.
selector.fit(X, y)

# Obtenemos máscara de variables seleccionadas.
mask = selector.get_support()

# Obtenemos nombres seleccionados.
selected_features = X.columns[mask]

# Mostramos variables.
print(selected_features.tolist())


### Interpretación

SelectKBest evalúa variables individualmente. Es útil para reducir dimensionalidad, aunque no considera interacciones entre variables.

## Bloque 6. Visualizar puntajes

Graficaremos los puntajes de las variables seleccionadas.

In [ ]:
# Creamos serie con scores.
scores = pd.Series(selector.scores_, index=X.columns).sort_values(ascending=False)

# Creamos figura.
plt.figure(figsize=(10,5))

# Graficamos top 10.
scores.head(10).plot(kind="bar")

# Agregamos título.
plt.title("Top 10 variables por SelectKBest")

# Rotamos etiquetas.
plt.xticks(rotation=45, ha="right")

# Mostramos gráfica.
plt.show()


### Interpretación

La gráfica permite justificar visualmente qué variables tienen mayor poder estadístico individual.

## Bloque 7. Comparar modelo con todas las variables vs seleccionadas

Entrenaremos regresión logística en ambos escenarios.

In [ ]:
# Separamos datos completos.
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)

# Creamos escalador.
scaler = StandardScaler()

# Escalamos datos completos.
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Creamos modelo completo.
model_all = LogisticRegression(max_iter=1000)

# Entrenamos modelo completo.
model_all.fit(X_train_scaled, y_train)

# Predecimos.
pred_all = model_all.predict(X_test_scaled)

# Calculamos accuracy.
acc_all = accuracy_score(y_test, pred_all)

# Creamos dataset con variables seleccionadas.
X_selected = X[selected_features]

# Separamos seleccionadas.
Xs_train, Xs_test, ys_train, ys_test = train_test_split(X_selected, y, test_size=0.3, random_state=42, stratify=y)

# Escalamos seleccionadas.
scaler_s = StandardScaler()
Xs_train_scaled = scaler_s.fit_transform(Xs_train)
Xs_test_scaled = scaler_s.transform(Xs_test)

# Creamos modelo seleccionado.
model_selected = LogisticRegression(max_iter=1000)

# Entrenamos modelo seleccionado.
model_selected.fit(Xs_train_scaled, ys_train)

# Predecimos.
pred_selected = model_selected.predict(Xs_test_scaled)

# Calculamos accuracy seleccionado.
acc_selected = accuracy_score(ys_test, pred_selected)

# Mostramos resultados.
print("Accuracy todas las variables:", acc_all)
print("Accuracy variables seleccionadas:", acc_selected)


### Interpretación

Si el modelo con menos variables mantiene el desempeño, ganamos simplicidad e interpretabilidad. Si baja mucho, la selección fue insuficiente.

## Bloque 8. Importancia con Random Forest

Usaremos un modelo de árboles para estimar importancia de variables.

In [ ]:
# Creamos Random Forest.
rf = RandomForestClassifier(n_estimators=200, random_state=42)

# Entrenamos Random Forest.
rf.fit(X, y)

# Creamos serie de importancias.
importances = pd.Series(rf.feature_importances_, index=X.columns).sort_values(ascending=False)

# Mostramos top 10.
print(importances.head(10))

# Creamos figura.
plt.figure(figsize=(10,5))

# Graficamos top 10.
importances.head(10).plot(kind="bar")

# Agregamos título.
plt.title("Top 10 variables por Random Forest")

# Rotamos etiquetas.
plt.xticks(rotation=45, ha="right")

# Mostramos gráfica.
plt.show()


### Interpretación

Random Forest captura relaciones más complejas que la correlación simple. Sirve como contraste para validar variables importantes.

## Cierre

La selección de características no solo busca mejorar accuracy; también busca modelos más interpretables, livianos y sostenibles.